In [2]:
from bellmira.llm_experiments.experiments import Experiments

In [1]:
from bellmira.llm_model.llm_model import LLMModel

c:\Users\X363244\AppData\Local\pypoetry\Cache\virtualenvs\bellmira-XxAPLhKE-py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import bellmira.utils.dict_logger

In [4]:
from bellmira.evaluators import (
    ModelContextLengthEvaluator, ModelParallelLoadEvaluator,
    ModelVisionEvaluator, ModelClassificationEvaluator,
    ModelTTFTEvaluator, ModelEmbeddingQualityEvaluator,
    ModelSummarizationEvaluator, ModelRegressionEvaluator,
)
from bellmira.utils.context_utils import contexts_from_bible, contexts_from_word_counts

# Variable Init

In [6]:

call_categorization_prompt = """
És um especialista de análise de conteúdos de chamadas de Call Center do Millennium BCP, um Banco de Portugal.

A tua função é categorizar a chamada com base num resumo dos conteúdos da chamada. A categoria deve estar associada a um serviço ou produto do Banco que o Cliente pretende utilizar ou obter esclarecimentos ou tem problemas. A categoria deve estar associada à necessidade do Cliente e não à forma que o Operador encontra para a resolver.

Quando a chamada tem várias categorias aplicáveis, escolhe aquela que mais se ajusta à operação que o Cliente pretende realizar ou produto/serviço que o Cliente refere ou assunto que pretende esclarecer.

Começa por determinar o motivo e objetivo inicial do Cliente e com base nesse motivo classifica a chamada.

Lista de categorias que podes utilizar:
* Transferências: Questões, dúvidas ou pedidos do Cliente relacionados com a realização de transferências de dinheiro entre contas.
* Cartões: Assuntos referentes a cartões de débito, cartões de crédito, cartões pré-pagos, virtuais (MB Net) incluindo ativação, bloqueio, ou dúvidas.
* Crédito: Dúvidas ou questões sobre produtos de crédito, como empréstimos pessoais, hipotecas, crédito habitação, ou linhas de crédito.
* Investimentos: Informações ou problemas relacionados com produtos de investimento, como ações, fundos de investimento, carteiras de títulos, ou obrigações.
* Poupanças: Questões relacionadas com contas poupança, depósitos a prazo, ou outros produtos de poupança.
* Contas: Questões sobre contas bancárias, como abertura, encerramento, manutenção de contas, ou temas associados como penhoras, habilitação de herdeiros, depósitos e levantamentos de numerário.
* Serviços Emergência: Pedidos do Cliente para assistência urgente, como bloqueio de contas ou cartões, devido a fraude, burlas, phishing.
* Pagamentos: Problemas ou dúvidas sobre o pagamento de serviços (como água, luz, internet), pagamentos de impostos ou ao estado, ou débitos diretos.
* MBway: Problemas ou pedidos relacionados com o serviço MBway.
* Atualização Dados: Questões ou pedidos para alteração ou atualização de dados pessoais ou de acesso. Inclui dados de acesso aos canais digitais do Banco, tal como alteração de telemóvel, código multicanal, ou dados Pessoais do Cliente como morada, documentos de identificação. 
* Site e Mobile: Problemas ou dúvidas gerais sobre o uso do site do banco ou da aplicação mobile, e que não se incluam no âmbito das restantes categorias.
* Pedido Contacto Sucursal: Quando o Cliente liga a solicitar contacto com uma Sucursal sem especificar problema e recusando-se a fazer, produto ou serviço especifico sobre o qual pretende esclarecimentos. Caso contrário a categorização deve ter em conta o verdadeiro tema que causou a chamada.
* Seguros: Pedidos ou questões relacionadas com seguros vida, automóvel, saúde, viagens ou outros. Tipicamente seguros da companhia Ocidental, do tipo Médis, Pétis, Homein. Também se incluem os seguros relativos aos produtos como seguros de cartão ou sobre o crédito de habitação, casos em que se sobrepõe a categorização de seguros.
* Outras - Quando nenhuma das anteriores é aplicável

Devolve apenas a categoria, utilizando uma das opções da lista acima. Escreve a opção escolhida tal como está escrita na lista. Não escrevas mais nada para além da categoria do problema.
"""

In [7]:
PROMPTS = [
    "Genesis 1: In the beginning God created the heavens and the earth. What does this tell us about the nature of creation?",
    "Genesis 2: And the Lord God formed man of the dust of the ground. How can this be interpreted in modern philosophy?",
    "Genesis 3: What is the symbolic significance of the Tree of Knowledge of Good and Evil?",
    "Genesis 4: What does the story of Cain and Abel teach about jealousy and responsibility?",
    "Genesis 5: What is the significance of the long lifespans recorded in the genealogies?",
    "Genesis 6: How does the story of Noah reflect themes of divine judgment and salvation?",
    "Genesis 7: Why does God instruct Noah to bring both clean and unclean animals?",
    "Genesis 8: What is the symbolism behind the dove returning with an olive leaf?",
    "Genesis 9: What is the meaning of God's covenant with Noah and the rainbow sign?",
    "Genesis 10: What does the Table of Nations reveal about ancient views of ethnicity and origins?",
    "Genesis 11: What does the Tower of Babel narrative teach about pride and human ambition?",
    "Genesis 12: What is the significance of God’s call to Abram to leave his homeland?",
    "Genesis 13: How does Abram's choice to let Lot choose land first demonstrate faith?",
    "Genesis 14: What is the role of Melchizedek, and how is he viewed in later biblical texts?",
    "Genesis 15: What is the importance of God’s covenant with Abram?",
    "Genesis 16: How does the story of Hagar explore issues of agency and divine justice?",
    "Genesis 17: What is the theological meaning of circumcision as a covenant sign?",
    "Genesis 18: How does Abraham's negotiation with God over Sodom reflect intercession?",
    "Genesis 19: What lessons can be drawn from the destruction of Sodom and Gomorrah?",
    "Genesis 20: How does Abraham’s repeated deception with Sarah highlight human weakness?",
    "Genesis 21: What is the significance of Isaac's birth to the Abrahamic promise?",
    "Genesis 22: How is Abraham’s willingness to sacrifice Isaac interpreted across traditions?",
    "Genesis 23: What does Abraham’s negotiation for Sarah’s burial site show about respect and legacy?",
    "Genesis 24: How does the story of Rebekah’s selection as Isaac’s wife illustrate divine guidance?",
    "Genesis 25: What is the significance of Jacob and Esau's struggle even before birth?",
    "Genesis 26: Why does Isaac repeat his father's lie about his wife being his sister?",
    "Genesis 27: What ethical questions are raised by Jacob’s deception of Isaac?",
    "Genesis 28: How should Jacob’s ladder dream be understood spiritually?",
    "Genesis 29: How does the story of Leah and Rachel reflect ancient marriage customs and rivalry?",
    "Genesis 30: What do the mandrake negotiations between Rachel and Leah tell us about desperation and power?",
    "Genesis 31: How does Laban’s pursuit of Jacob reflect broken family dynamics?",
    "Genesis 32: What is the meaning of Jacob wrestling with the angel?",
    "Genesis 33: How does Jacob’s reconciliation with Esau contrast with earlier deception?",
    "Genesis 34: What moral and political implications arise from the story of Dinah?",
    "Genesis 35: What is the significance of Jacob’s name change to Israel?",
    "Genesis 36: How do the genealogies of Esau (Edom) serve the larger biblical narrative?",
    "Genesis 37: What psychological dynamics are at play in Joseph’s brothers’ jealousy?",
    "Genesis 38: What does the story of Judah and Tamar suggest about justice and redemption?",
    "Genesis 39: How does Joseph’s resistance to Potiphar’s wife model moral integrity?",
    "Genesis 40: What is the role of dreams in Joseph’s story and in Genesis as a whole?",
    "Genesis 41: How does Joseph’s rise to power in Egypt highlight providence and preparation?",
    "Genesis 42: How do Joseph’s brothers’ reactions to famine reflect guilt and repentance?",
    "Genesis 43: What does Judah’s pledge for Benjamin show about transformation and leadership?",
    "Genesis 44: Why does Joseph test his brothers before revealing himself?",
    "Genesis 45: What theological themes emerge when Joseph says, 'God sent me before you'?",
    "Genesis 46: What is the significance of Jacob’s migration to Egypt?",
    "Genesis 47: How does Joseph’s land policy in Egypt raise questions about power and control?",
    "Genesis 48: What is the meaning of Jacob’s blessing of Ephraim over Manasseh?",
    "Genesis 49: How do Jacob’s prophetic blessings shape the destiny of the tribes of Israel?",
    "Genesis 50: What does Joseph’s final statement, 'You meant it for evil, but God meant it for good,' reveal about divine providence?"
]

# QA - Evaluators init

In [8]:
BIBLE_PATH = "/workspaces/BeLLMira/resources/data/text/bible.txt"

# Build context dicts using the helper — no more bible_path / chapter_numbers in evaluator args
context_length_contexts = contexts_from_bible(
    bible_path=BIBLE_PATH,
    chapter_numbers=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 18, 19, 20, 21, 22, 23, 24, 25, 30, 35, 40],
)
parallel_load_contexts = contexts_from_bible(
    bible_path=BIBLE_PATH,
    chapter_numbers=[2, 3, 5, 9, 14],
)

evaluator_args = {
    "prompts": PROMPTS[:2],
    "contexts": context_length_contexts,
}
parallel_load_evaluator_args = {
    "prompts": PROMPTS,
    "contexts": parallel_load_contexts,
    "concurrency_levels": [2, 4, 8, 16, 32, 64],
}
vision_evaluator_args = {}

In [ ]:
experiments_list = [
            {
                "model": "Qwen 2",
                "url": "https://millmchatqwen2-backend.dat-aip-millm.qa.mbcp.cloud/",
                "env": "QA",
                "vllm_version": "0.9.2",
                "evaluator": ModelContextLengthEvaluator,
                "evaluator_args": evaluator_args,
                "output_filename": "exp_context_length_results.csv",
            },
            {
                "model": "Qwen 2.5",
                "url": "https://millmchatqwen25-backend.dat-aip-millm.qa.mbcp.cloud/",
                "env": "QA",
                "vllm_version": "0.9.2",
                "evaluator": ModelContextLengthEvaluator,
                "evaluator_args": evaluator_args,
                "output_filename": "exp_context_length_results.csv",
            },
            {
                "model": "Llama3.1",
                "url": "https://millmchatllama31-backend.dat-aip-millm.qa.mbcp.cloud/",
                "env": "QA",
                "vllm_version": "0.9.2",
                "evaluator": ModelContextLengthEvaluator,
                "evaluator_args": evaluator_args,
                "output_filename": "exp_context_length_results.csv",
            },
            {
                "model": "Llama3",
                "url": "https://millmchat-backend.dat-aip-millm.qa.mbcp.cloud/",
                "env": "QA",
                "vllm_version": "0.9.2",
                "evaluator": ModelContextLengthEvaluator,
                "evaluator_args": evaluator_args,
                "output_filename": "exp_context_length_results.csv",
            },
]

In [17]:
experiments_list = [
            {
                "model": "Qwen 2.5 IN",
                "url": "https://millmchatqwen25-backend.dat-aip-millm.in.mbcp.cloud/",
                "env": "QA",
                "vllm_version": "0.9.2",
                "evaluator": ModelContextLengthEvaluator,
                "evaluator_args": evaluator_args,
                "output_filename": "exp_context_length_results.csv",
            },
            {
                "model": "Qwen 2.5 QA",
                "url": "https://millmchatqwen25-backend.dat-aip-millm.qa.mbcp.cloud/",
                "env": "QA",
                "vllm_version": "0.9.2",
                "evaluator": ModelContextLengthEvaluator,
                "evaluator_args": evaluator_args,
                "output_filename": "exp_context_length_results.csv",
            }
]

In [18]:
experiments = Experiments(experiments_list, vllm_version="0.9.2")
experiments.create_experiments()
experiments.launch_experiments()

Running experiment for model: Qwen 2.5 IN
{'model': 'Qwen 2.5 IN', 'url': 'https://millmchatqwen25-backend.dat-aip-millm.in.mbcp.cloud/', 'env': 'QA', 'vllm_version': '0.9.2', 'evaluator': <class 'bellmira.evaluators.model_context_length_evaluator.ModelContextLengthEvaluator'>, 'evaluator_args': {'bible_path': '/workspaces/BeLLMira/resources/data/text/bible.txt', 'prompts': ['Genesis 1: In the beginning God created the heavens and the earth. What does this tell us about the nature of creation?', 'Genesis 2: And the Lord God formed man of the dust of the ground. How can this be interpreted in modern philosophy?'], 'chapter_numbers': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 18, 19, 20, 21, 22, 23, 24, 25, 30, 35, 40], 'url': 'https://millmchat-backend.dat-aip-millm.qa.mbcp.cloud/'}, 'output_filename': 'exp_context_length_results.csv'}
Running experiment for model: Qwen 2.5 QA
{'model': 'Qwen 2.5 QA', 'url': 'https://millmchatqwen25-backend.dat-aip-millm.qa.mbcp.cloud/', 'env': 'Q